In [1]:
"""
07_segmentation.py
--------------------
Phase 9: groups customers into High/Medium/Low CLV tiers using
K-means clustering, instead of arbitrary percentile cutoffs.

Clustering uses predicted_clv plus two supporting behavioral
features (frequency, monetary_total) so the tiers reflect actual
behavioral groupings, not just a single number sliced into thirds.

Reads:
    data/processed/predicted_clv.csv
    data/processed/customer_features.csv   (for frequency, monetary_total)

Writes:
    data/processed/segmented_customers.csv
"""

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ----------------------------------------------------------------
# 1. LOAD AND COMBINE DATA
# ----------------------------------------------------------------
predictions = pd.read_csv("data/processed/predicted_clv.csv")
features = pd.read_csv("data/processed/customer_features.csv")

df = predictions.merge(
    features[["customer_id", "frequency", "monetary_total"]],
    on="customer_id",
)
print(f"Loaded {len(df)} customers")

# ----------------------------------------------------------------
# 2. SCALE FEATURES FOR CLUSTERING
#    K-means is distance-based, so features need to be on a
#    comparable scale - otherwise predicted_clv (in the thousands)
#    would completely dominate frequency (single digits).
# ----------------------------------------------------------------
cluster_features = ["predicted_clv", "frequency", "monetary_total"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[cluster_features])

# ----------------------------------------------------------------
# 3. RUN K-MEANS (k=3: High / Medium / Low)
# ----------------------------------------------------------------
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

# ----------------------------------------------------------------
# 4. LABEL CLUSTERS BY THEIR ACTUAL MEAN PREDICTED CLV
#    K-means assigns arbitrary cluster numbers (0, 1, 2) - this
#    step maps them to meaningful labels based on which cluster
#    actually has the highest/lowest average value.
# ----------------------------------------------------------------
cluster_means = df.groupby("cluster")["predicted_clv"].mean().sort_values()
tier_labels = {
    cluster_means.index[0]: "Low",
    cluster_means.index[1]: "Medium",
    cluster_means.index[2]: "High",
}
df["clv_tier"] = df["cluster"].map(tier_labels)

# ----------------------------------------------------------------
# 5. SANITY CHECKS
# ----------------------------------------------------------------
print("\nCustomer count per tier:")
print(df["clv_tier"].value_counts())

print("\nAverage predicted CLV per tier (should be cleanly separated, in order):")
print(df.groupby("clv_tier")["predicted_clv"].mean().round(2).sort_values())

print("\nAcquisition channel mix within each tier (preview of the chi-square test to come):")
channel_mix = pd.crosstab(df["clv_tier"], df["acquisition_channel"], normalize="index")
print((channel_mix * 100).round(1))

# ----------------------------------------------------------------
# 6. SAVE OUTPUT
# ----------------------------------------------------------------
output_cols = ["customer_id", "acquisition_channel", "age_band", "region", "predicted_clv", "clv_tier"]
df[output_cols].to_csv("data/processed/segmented_customers.csv", index=False)
print("\nSaved: data/processed/segmented_customers.csv")

Loaded 4000 customers

Customer count per tier:
clv_tier
Medium    1928
Low       1749
High       323
Name: count, dtype: int64

Average predicted CLV per tier (should be cleanly separated, in order):
clv_tier
Low        2796.55
Medium     4113.97
High      14109.35
Name: predicted_clv, dtype: float64

Acquisition channel mix within each tier (preview of the chi-square test to come):
acquisition_channel  Email  Organic  Paid Search  Referral  Social
clv_tier                                                          
High                   0.0      0.3          0.0      99.7     0.0
Low                   10.1     11.7         45.7       1.8    30.8
Medium                31.5     18.9         34.6       2.3    12.7

Saved: data/processed/segmented_customers.csv
